# Tool Use (Function Calling)

The **Tool Use** pattern enables an LLM to interact with external APIs, databases, and services by deciding autonomously when and how to call specific functions. It bridges the gap between the LLM's reasoning capabilities and real-world execution.

The process follows a well-defined loop:
1. **Tool Definition** — Describe available tools to the LLM (name, purpose, parameter schema).
2. **LLM Decision** — The model receives the user query and decides if a tool call is necessary.
3. **Structured Call Generation** — The model outputs a structured object specifying the tool name and arguments.
4. **Tool Execution** — Your code executes the actual function with those arguments.
5. **Result Injection** — The tool result is returned to the model as context.
6. **Final Response** — The model produces a final answer incorporating the tool's output.

This loop repeats until the model produces a response without requesting any further tool calls.

**Use cases:** financial data retrieval, inventory queries, real-time search, code execution, sending notifications, controlling external systems.

## Implementation with Flyte v2 + the Agent harness

Flyte v2 ships a batteries-included agent loop — `flyte.ai.agents.Agent` — that drives the LLM ↔ tool conversation for you. You declare tools as plain Python functions decorated with `@tool`; the harness generates their JSON schema from type hints + docstrings, runs the loop, executes tool calls (in parallel by default), and returns a typed `AgentResult`. The hand-rolled `while` loop and `_execute_tool` dispatcher that earlier Flyte-v2 examples needed disappear entirely.

#### LangChain vs Flyte v2 + Agent harness

| Aspect | LangChain (`AgentExecutor`) | Flyte v2 + `Agent` harness |
|--------|----------------------------|----------------------------|
| **Tool definition** | `@langchain_tool` decorator | `@tool` — schema inferred from type hints + docstrings |
| **Agent loop** | `AgentExecutor.ainvoke` (black box) | `Agent.run` — managed loop, every tool call traced |
| **Checkpointing** | None | Tools run as traced / `@env.task` steps — durable, resumable |
| **State** | In-process `AgentScratchpad` | Typed `AgentResult` (`summary`, `error`, `attempts`) |
| **Observability** | Verbose logging | Nested tool-call sub-actions in the Flyte UI |
| **Execution** | In-process only | Local or remote (containers on Kubernetes) |

### 1. Install dependencies

In [ ]:
!uv pip install flyte litellm -U

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [3]:
!flyte start devbox

flyte-devbox
  Waiting for flyte cluster to be ready ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:02:115m 83% 0:02:11
╭──────────────────────────────── Flyte Devbox ────────────────────────────────╮
│ Flyte devbox cluster is ready!                                               │
│                                                                              │
│   🚀 UI:             ]8;id=4839095;http://localhost:30080/v2\http://localhost:30080/v2]8;;\                               │
│   🐳 Image Registry: localhost:30000                                         │
╰──────────────────────────────────────────────────────────────────────────────╯


### 2. Export your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...


### 3. Import dependencies and configure the Flyte TaskEnvironment

In [ ]:
import os
from datetime import timedelta
from typing import Any, TypedDict

import flyte
import flyte.report
from flyte.ai.agents import Agent, AgentResult, tool

flyte.init_from_config()

# The Agent's default LLM callback uses litellm, which reads ANTHROPIC_API_KEY
# from the environment — injected here by the Flyte secret.
_image = (
    flyte.Image.from_debian_base(name="tool-use-agent", python_version=(3, 12))
    .with_pip_packages("litellm")
)

tool_env = flyte.TaskEnvironment(
    name="tool_use_env",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the tools

Tools are plain Python functions decorated with `@tool` from `flyte.ai.agents`. The decorator inspects each function's type hints and docstring to build the JSON schema the LLM sees — there is no manual schema wiring and no custom registry to maintain. This example builds a **financial research assistant** that can look up stock prices, value a portfolio, and search financial news.

Each function below becomes a tool the agent can call:

In [33]:
class Holding(TypedDict):
    ticker: str
    shares: float


@tool
def get_stock_price(ticker: str) -> dict:
    """Get the current price, daily change %, and market cap for a stock ticker.

    Args:
        ticker: The stock ticker symbol, e.g. 'AAPL' for Apple.
    """
    simulated = {
        "AAPL": {"price": 178.15, "change_pct": 1.23, "market_cap_billions": 2780},
        "MSFT": {"price": 425.50, "change_pct": -0.45, "market_cap_billions": 3160},
        "GOOGL": {"price": 175.30, "change_pct": 0.87, "market_cap_billions": 2190},
        "NVDA": {"price": 875.40, "change_pct": 3.12, "market_cap_billions": 2150},
    }
    ticker = ticker.upper()
    if ticker not in simulated:
        return {"error": f"Ticker '{ticker}' not found. Available: {list(simulated.keys())}"}
    return {"ticker": ticker, **simulated[ticker]}


@tool
def calculate_portfolio_value(holdings: list[Holding]) -> dict:
    """Calculate the total current market value of a stock portfolio.

    Args:
        holdings: List of positions, each with a ticker symbol and number of shares.
    """
    prices = {"AAPL": 178.15, "MSFT": 425.50, "GOOGL": 175.30, "NVDA": 875.40}
    positions, total = [], 0.0
    for h in holdings:
        ticker = h["ticker"].upper()
        price = prices.get(ticker)
        if price is None:
            return {"error": f"Unknown ticker: {ticker}"}
        value = price * h["shares"]
        total += value
        positions.append({"ticker": ticker, "shares": h["shares"], "value": round(value, 2)})
    largest = max(positions, key=lambda p: p["value"])
    return {"total_value": round(total, 2), "positions": positions, "largest_position": largest}


@tool
def search_financial_news(query: str, max_results: int = 3) -> dict:
    """Search for recent financial news articles matching the query.

    Args:
        query: Search query string for financial news.
        max_results: Maximum number of articles to return (1-10).
    """
    articles = [
        {"title": "Apple Reports Record Q4 Revenue of $94.9B",
         "summary": "Apple exceeded analyst expectations with strong iPhone and Services growth.",
         "date": "2024-11-01"},
        {"title": "Microsoft Cloud Revenue Surges 33% on AI Demand",
         "summary": "Azure growth accelerated as enterprise AI adoption drives cloud spending.",
         "date": "2024-10-30"},
        {"title": "NVIDIA Data Center Revenue Hits $30B in Q3",
         "summary": "GPU demand from AI hyperscalers continues to outpace supply.",
         "date": "2024-11-20"},
        {"title": "Alphabet Beats Estimates with 15% Revenue Growth",
         "summary": "Google Search and YouTube both showed strong advertising recovery.",
         "date": "2024-10-29"},
    ]
    query_lower = query.lower()
    matched = [a for a in articles if any(
        w in a["title"].lower() or w in a["summary"].lower()
        for w in query_lower.split()
    )]
    return {"articles": (matched or articles)[:max_results], "total_found": len(matched or articles)}

### 5. Build the agent

#### From manual loop to Agent harness

The earlier Flyte-v2 version of this section was ~120 lines: a `SYSTEM_PROMPT`, a `@flyte.trace`-d `_execute_tool` dispatcher, a `_call_llm` helper, and an explicit `for turn in range(max_turns)` loop that parsed `stop_reason`, converted SDK content blocks to dicts, and threaded `messages` by hand.

`Agent` collapses all of it into a single declarative object. You get for free:
- the **LLM ↔ tool loop** with a `max_turns` guard,
- **parallel tool execution** (`parallel_tool_calls=True`),
- per-tool **tracing** in the Flyte UI,
- a typed **`AgentResult`** (`.summary`, `.error`, `.attempts`).

In [ ]:
SYSTEM_PROMPT = """\
You are a financial research assistant with access to real-time stock data and news.
Use the available tools to retrieve accurate data before answering.
Always cite the specific data you retrieved when making claims.
Be concise and factual. If a tool returns an error, explain the limitation clearly."""

finance_agent = Agent(
    name="finance-helper",
    instructions=SYSTEM_PROMPT,
    model="claude-sonnet-4-6",
    tools=[get_stock_price, calculate_portfolio_value, search_financial_news],
    max_turns=10,
)

In [ ]:
@tool_env.task(
    retries=3,
    timeout=timedelta(minutes=15),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def tool_use_agent(query: str) -> str:
    """Financial research agent.

    The Agent harness drives the LLM ↔ tool loop; each tool call appears as a
    nested, traced sub-action in the Flyte UI. We surface the final answer in the
    report tab and return it as the task output.
    """
    result: AgentResult = await finance_agent.run.aio(query)

    answer = result.summary or result.error or ""
    await flyte.report.replace.aio(
        "<html><body style='font-family:sans-serif;max-width:900px;margin:auto;padding:1.5em'>"
        "<h1>Tool Use Agent — Final Answer</h1>"
        "<pre style='background:#e8f5e9;padding:1em;border-radius:4px;white-space:pre-wrap'>"
        + answer.replace("&", "&amp;").replace("<", "&lt;") +
        "</pre></body></html>"
    )
    await flyte.report.flush.aio()

    if result.error:
        raise RuntimeError(result.error)
    return result.summary

### 6. Run locally

In [ ]:
run = flyte.run(
    tool_use_agent,
    query="What is Apple's current stock price and how has it been performing recently?",
)
run.wait()
print(run.outputs()[0])
print(run.url)

### Adding real tools

To connect real APIs, replace the simulated functions with actual HTTP calls. The agent loop requires no changes — only the tool implementations change:

In [ ]:
# Example: replacing get_stock_price with a real Alpha Vantage call
# (requires ALPHA_VANTAGE_API_KEY secret)
import httpx

async def get_stock_price_real(ticker: str) -> dict:
    """Production implementation using Alpha Vantage API."""
    api_key = os.environ["ALPHA_VANTAGE_API_KEY"]
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={ticker}&apikey={api_key}"
    async with httpx.AsyncClient() as client:
        resp = await client.get(url, timeout=10.0)
        resp.raise_for_status()
        data = resp.json().get("Global Quote", {})
        return {
            "ticker": ticker,
            "price": float(data.get("05. price", 0)),
            "change_pct": float(data.get("10. change percent", "0%").strip("%")),
        }

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_tool_use = flyte.TaskEnvironment(
    name="tool_use_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)

### Parallel tool execution — built in

When the LLM requests several tools in one turn, `Agent` runs them concurrently by default (`parallel_tool_calls=True`). The earlier example hand-rolled this with `asyncio.gather`; with the harness there is nothing to wire up. To force sequential execution — e.g. when tools share mutable state — construct the agent with `parallel_tool_calls=False`.